# Timestep Stability

Inspect the RAPID timestep tails directly from the prepared subreaches.
The notebook focuses on the actual subreach that sets each state's `dt_min` and `dt_max`.

## Imports

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "RAPID" / "src" / "rapid_tools").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing RAPID/src/rapid_tools")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAPID_SRC = REPO_ROOT / "RAPID" / "src"

if str(RAPID_SRC) not in sys.path:
    sys.path.append(str(RAPID_SRC))

if "rapid_tools" in sys.modules:
    importlib.reload(sys.modules["rapid_tools"])

from rapid_tools import (
    build_timestep_tail_frame,
    plot_effective_length_vs_dt,
    plot_timestep_tail_suite,
)

## Configuration

In [2]:
EXAMPLE_IDS = ["21"]
MODE = "indep"  # or "compound"
STATE_IDS = None  # e.g. ["S000_base", "S004_unit_5"]
INCLUDE_BASE = True

## Load Tail Culprits

In [3]:
tail_df = build_timestep_tail_frame(
    EXAMPLE_IDS,
    mode=MODE,
    state_ids=STATE_IDS,
    include_base=INCLUDE_BASE,
)

tail_df

,example_id,mode,state_id,state_label,prep_status,tail,state_dt_value,rapid_k,parent_link_id,id_link,subreach_index,rapid_effective_length_m,rapid_celerity_mps,rapid_width_m,rapid_subreach_width_m,slope_used,tail_label
0,21,indep,S000_base,21 · S000_base,failed,dt_max,626.343000,347.968339,38,38,1,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,Smallest dt_max culprit
1,21,indep,S000_base,21 · S000_base,failed,dt_min,626.949000,3134.743683,4,4,4,219.432058,0.070000,254.374260,254.374260,9.074820e-05,Largest dt_min culprit
2,21,indep,S001_unit_8,21 · S001_unit_8,failed,dt_max,626.343000,347.968339,35,35,1,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,Smallest dt_max culprit
3,21,indep,S001_unit_8,21 · S001_unit_8,failed,dt_min,626.949000,3134.743683,4,4,4,219.432058,0.070000,254.374260,254.374260,9.074820e-05,Largest dt_min culprit
4,21,indep,S002_unit_15,21 · S002_unit_15,failed,dt_max,626.343000,347.968339,38,38,1,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,Smallest dt_max culprit
5,21,indep,S002_unit_15,21 · S002_unit_15,failed,dt_min,626.949000,3134.743683,4,4,4,219.432058,0.070000,254.374260,254.374260,9.074820e-05,Largest dt_min culprit
6,21,indep,S003_unit_13,21 · S003_unit_13,failed,dt_max,603.483000,335.268399,77,77,1,200.000000,0.596537,1204.362093,1204.362093,8.882552e-04,Smallest dt_max culprit
7,21,indep,S003_unit_13,21 · S003_unit_13,failed,dt_min,626.949000,3134.743683,4,4,4,219.432058,0.070000,254.374260,254.374260,9.074820e-05,Largest dt_min culprit
8,21,indep,S004_unit_5,21 · S004_unit_5,failed,dt_max,626.343000,347.968339,35,35,1,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,Smallest dt_max culprit
9,21,indep,S004_unit_5,21 · S004_unit_5,failed,dt_min,757.297000,3786.484398,13,13,1,265.053908,0.070000,264.348716,264.348716,5.424064e-05,Largest dt_min culprit


## Tail Table

In [20]:
626.343 * 0.065 / 0.2

203.56147499999997

In [16]:
D = tail_df[tail_df['tail'] == 'dt_min'].copy()
2*(D['rapid_effective_length_m'] / D['rapid_celerity_mps'])*0.1

D['test'] = 2*(218 / D['rapid_celerity_mps'])*0.1
D[['state_dt_value', 'test']]

,state_dt_value,test
1,626.949000,622.857143
3,626.949000,622.857143
5,626.949000,622.857143
7,626.949000,622.857143
9,757.297000,622.857143
11,626.949000,622.857143
13,626.949000,622.857143
15,626.949000,622.857143
17,626.949000,622.857143
19,626.949000,622.857143


In [4]:
display(
    tail_df[[
        c for c in [
            "example_id",
            "state_id",
            "tail",
            "state_dt_value",
            "parent_link_id",
            "id_link",
            "subreach_index",
            "rapid_k",
            "rapid_effective_length_m",
            "rapid_celerity_mps",
            "rapid_width_m",
            "rapid_subreach_width_m",
            "slope_used",
            "prep_status",
        ] if c in tail_df.columns
    ]].sort_values(["tail", "state_dt_value"], ascending=[True, True])
)

,example_id,state_id,tail,state_dt_value,parent_link_id,id_link,subreach_index,rapid_k,rapid_effective_length_m,rapid_celerity_mps,rapid_width_m,rapid_subreach_width_m,slope_used,prep_status
6,21,S003_unit_13,dt_max,603.483000,77,77,1,335.268399,200.000000,0.596537,1204.362093,1204.362093,8.882552e-04,failed
0,21,S000_base,dt_max,626.343000,38,38,1,347.968339,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,failed
2,21,S001_unit_8,dt_max,626.343000,35,35,1,347.968339,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,failed
4,21,S002_unit_15,dt_max,626.343000,38,38,1,347.968339,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,failed
8,21,S004_unit_5,dt_max,626.343000,35,35,1,347.968339,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,failed
10,21,S005_unit_24,dt_max,626.343000,38,38,1,347.968339,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,failed
12,21,S006_unit_11,dt_max,626.343000,35,35,1,347.968339,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,failed
14,21,S007_unit_18,dt_max,626.343000,38,38,1,347.968339,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,failed
16,21,S008_unit_10,dt_max,626.343000,35,35,1,347.968339,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,failed
18,21,S009_unit_20,dt_max,626.343000,38,38,1,347.968339,200.000000,0.574765,1139.033276,1139.033276,8.882552e-04,failed


## Effective Length vs Both Tails

In [5]:
fig = plot_timestep_tail_suite(tail_df)
fig.show()

## Effective Length vs `dt_min`

In [6]:
fig = plot_effective_length_vs_dt(tail_df, tail="dt_min")
fig.show()

## Effective Length vs `dt_max`

In [7]:
fig = plot_effective_length_vs_dt(tail_df, tail="dt_max")
fig.show()